# LeetCode 4060 — Count Evenly Good Integers (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4060 (trên LeetCode bài này thuộc nhóm Premium).

Đây là bài nhập môn tuyệt vời cho **quy hoạch động theo chữ số (digit DP)** — kỹ thuật đếm các số trong một khoảng khổng lồ (`10^15`) thỏa một tính chất về chữ số. Ta sẽ học:

1. Mẹo **`F(r) - F(l - 1)`**: đếm trên `[l, r]` = đếm trên `[1, r]` trừ đếm trên `[1, l - 1]`.
2. Digit DP với ba trạng thái: **vị trí**, **tính chẵn lẻ**, **"còn bám sát cận trên" (tight)**, cùng cờ **"đã bắt đầu"** để xử lý số 0 ở đầu.
3. Một lối tắt tổ hợp: **đếm bằng (tổng + tổng có dấu) / 2**, và phát hiện bất ngờ: với số có từ 2 chữ số trở lên, **đúng một nửa** là "tốt".

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc chữ số của một số, kiểm tra tính chất "số chữ số chẵn là số chẵn".
2. Brute Force và vì sao không dùng được với `10^15`.
3. Digit DP: tư duy "điền chữ số từ trái sang phải", trạng thái `tight` và `started`, memoization.
4. Đếm tổ hợp với mẹo dấu `±1` (tách số chẵn/lẻ phần tử).
5. Lỗi hay gặp (số 0 ở đầu, `l - 1`, ...), mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai số nguyên `l <= r`. Một số nguyên dương được gọi là **evenly good** (tạm dịch: "tốt đều") nếu nó có **số lượng chữ số chẵn là một số chẵn** (0, 2, 4, ... chữ số chẵn). Chữ số chẵn là `0, 2, 4, 6, 8`.

Đếm số lượng số evenly good trong đoạn `[l, r]`.

---

## Ví dụ 1

```python
l = 18, r = 22   ->   3
```

| số | chữ số chẵn | số lượng | tốt? |
|---:|---|---:|:---:|
| 18 | 8 | 1 | |
| 19 | — | 0 | ✔ |
| 20 | 2, 0 | 2 | ✔ |
| 21 | 2 | 1 | |
| 22 | 2, 2 | 2 | ✔ |

## Ví dụ 2

```python
l = 98, r = 101   ->   2
```

`99` (0 chữ số chẵn) và `100` (hai số 0). `98` có 1 (số 8), `101` có 1 (số 0).

## Ví dụ 3

```python
l = 1, r = 10   ->   5
```

`1, 3, 5, 7, 9` (0 chữ số chẵn). `10` có 1 chữ số chẵn (số 0).

---

## Ràng buộc

```text
1 <= l <= r <= 10^15
```

# 2. Kiến thức nền tảng

## 2.1. Lấy chữ số của một số

Cách gọn nhất: `str(x)` rồi duyệt từng ký tự. (Cách "toán học" `% 10`, `// 10` cũng được.)

In [1]:
def count_even_digits(x: int) -> int:
    return sum(1 for ch in str(x) if int(ch) % 2 == 0)


def is_evenly_good(x: int) -> bool:
    return count_even_digits(x) % 2 == 0


for x in [18, 19, 20, 100, 101, 2468]:
    print(x, count_even_digits(x), is_evenly_good(x))

18 1 False
19 0 True
20 2 True
100 2 True
101 1 False
2468 4 True


## 2.2. Mẹo `F(r) - F(l - 1)`

Gọi `F(x)` = số lượng số tốt trong `[1, x]`. Khi đó:

```text
số lượng trong [l, r] = F(r) - F(l - 1)
```

Nhờ vậy ta chỉ cần giải bài "đếm từ 1 tới `x`" — một cận trên duy nhất. Quy ước `F(0) = 0`.

## 2.3. `functools.lru_cache`

Ghi nhớ kết quả của hàm theo tham số — mỗi trạng thái chỉ tính một lần.

# 3. Cách 1 — Brute Force

In [2]:
def count_bruteforce(l: int, r: int) -> int:
    return sum(is_evenly_good(x) for x in range(l, r + 1))


print(count_bruteforce(18, 22), count_bruteforce(98, 101), count_bruteforce(1, 10))

3 2 5


Duyệt `10^15` số là không tưởng (hàng chục năm). Nhưng hãy dùng brute force để **quan sát**:

In [3]:
for d in range(1, 6):
    lo, hi = 10 ** (d - 1), 10 ** d - 1
    good = count_bruteforce(lo, hi)
    print(f"{d} chữ số: {good:>6} / {hi - lo + 1:>6} số là tốt")

1 chữ số:      5 /      9 số là tốt
2 chữ số:     45 /     90 số là tốt
3 chữ số:    450 /    900 số là tốt
4 chữ số:   4500 /   9000 số là tốt
5 chữ số:  45000 /  90000 số là tốt


Với số có từ 2 chữ số trở lên, **đúng một nửa** là tốt! Ta sẽ hiểu vì sao ở mục 6. Trước hết, học kỹ thuật tổng quát: digit DP.

---

# 4. Digit DP: điền chữ số từ trái sang phải

Muốn đếm số `y` trong `[1, x]` thỏa tính chất, hãy tưởng tượng ta **điền từng chữ số** của `y` từ trái sang phải, với cùng độ dài như `x` (cho phép số 0 ở đầu để biểu diễn số ngắn hơn).

Ví dụ `x = 22` (hai chữ số `2, 2`). Khi điền chữ số đầu tiên:

- chọn `0` hoặc `1` (nhỏ hơn `2`) → chữ số sau được chọn **tự do** `0..9`;
- chọn `2` (bằng `2`) → chữ số sau **bị chặn** bởi `2` (chỉ `0..2`), nếu không sẽ vượt `x`.

Trạng thái khi đang ở vị trí `i`:

| Biến | Ý nghĩa |
|---|---|
| `i` | đang điền chữ số thứ `i` |
| `par` | tính chẵn lẻ của **số chữ số chẵn** đã điền (0 = chẵn) |
| `tight` | các chữ số đã điền có **trùng hệt** tiền tố của `x` không (nếu có, chữ số tiếp theo không được vượt chữ số của `x`) |
| `started` | đã điền chữ số khác 0 chưa (chưa thì các số 0 hiện tại là **số 0 ở đầu**, không phải chữ số thật) |

Cờ `started` rất quan trọng: số `7` khi viết dưới dạng `07` có một số `0` ở đầu — nhưng số 0 đó **không** phải chữ số của `7`, không được đếm là chữ số chẵn.

Kết thúc (đã điền hết): đếm 1 nếu `started` (số khác 0) và `par == 0`.

In [4]:
from functools import lru_cache


def F_digit_dp(x: int) -> int:
    if x <= 0:
        return 0
    digits = list(map(int, str(x)))
    n = len(digits)

    @lru_cache(maxsize=None)
    def go(i: int, par: int, tight: bool, started: bool) -> int:
        if i == n:
            return 1 if started and par == 0 else 0
        limit = digits[i] if tight else 9
        total = 0
        for d in range(limit + 1):
            now_started = started or d > 0
            new_par = par ^ (1 if now_started and d % 2 == 0 else 0)   # số 0 ở đầu không tính
            total += go(i + 1, new_par, tight and d == limit, now_started)
        return total

    return go(0, 0, True, False)


def count_digit_dp(l: int, r: int) -> int:
    return F_digit_dp(r) - F_digit_dp(l - 1)


print(count_digit_dp(18, 22), count_digit_dp(98, 101), count_digit_dp(1, 10))
print("[1, 10^15]:", count_digit_dp(1, 10**15))

3 2 5
[1, 10^15]: 500000000000000


## Mô phỏng `F(22)`

`digits = [2, 2]`. Ở vị trí 0 (`tight = True`, giới hạn `2`):

| chữ số đầu | started | par sau | tight sau | vị trí 1: chữ số được chọn | số tốt thu được |
|---|:---:|---:|:---:|---|---|
| 0 (số 0 ở đầu) | ✗ | 0 | ✗ | `0..9` → số một chữ số `0..9` | `1, 3, 5, 7, 9` → 5 |
| 1 | ✔ | 0 | ✗ | `0..9`, cần chữ số **lẻ** | `11, 13, 15, 17, 19` → 5 |
| 2 | ✔ | 1 | ✔ | `0..2`, cần chữ số **chẵn** | `20, 22` → 2 |

`F(22) = 12`. Tương tự `F(17) = 9` (thử tự tính: `1,3,5,7,9,11,13,15,17`), nên đáp án ví dụ 1 là `12 - 9 = 3` ✔.

In [5]:
print("F(22) =", F_digit_dp(22), "| F(17) =", F_digit_dp(17), "| hiệu =", F_digit_dp(22) - F_digit_dp(17))
print("liệt kê [1, 22]:", [y for y in range(1, 23) if is_evenly_good(y)])

F(22) = 12 | F(17) = 9 | hiệu = 3
liệt kê [1, 22]: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 20, 22]


Số trạng thái: `16 vị trí × 2 × 2 × 2`, mỗi trạng thái thử ≤ 10 chữ số → vài trăm phép tính. Cực nhanh.

---

# 5. Cách 2 — Digit DP lặp (không đệ quy)

Có thể viết DP theo chiều xuôi: giữ `cnt[(par, tight, started)]` = số tiền tố đã điền ứng với trạng thái đó, rồi lần lượt mở rộng từng chữ số. Cách này tránh đệ quy và cho thấy rõ "số tiền tố" chảy qua các trạng thái.

In [6]:
def F_iterative(x: int) -> int:
    if x <= 0:
        return 0
    digits = list(map(int, str(x)))
    cnt = {(0, True, False): 1}
    for digit in digits:
        new = {}
        for (par, tight, started), ways in cnt.items():
            limit = digit if tight else 9
            for d in range(limit + 1):
                ns = started or d > 0
                np_ = par ^ (1 if ns and d % 2 == 0 else 0)
                key = (np_, tight and d == limit, ns)
                new[key] = new.get(key, 0) + ways
        cnt = new
    return sum(w for (par, _, started), w in cnt.items() if started and par == 0)


print(F_iterative(22), F_iterative(10**15))

12 500000000000000


# 6. Cách 3 — Đếm tổ hợp với mẹo dấu `±1`

## 6.1. Mẹo: số cách có **số chẵn** phần tử "đặc biệt"

Giả sử có nhiều vị trí, mỗi vị trí chọn một chữ số; gọi `E` = số chữ số chẵn được chọn. Ta muốn đếm các cách có `E` chẵn.

- **Tổng** số cách: `T = Π (số lựa chọn ở mỗi vị trí)`.
- **Tổng có dấu**: `D = Σ (-1)^E` = `Π (số lựa chọn lẻ - số lựa chọn chẵn)` (khai triển tích).

Khi đó: `số cách E chẵn = (T + D) / 2`, `số cách E lẻ = (T - D) / 2`.

## 6.2. Áp dụng cho số có đúng `d` chữ số

- Chữ số đầu: `1..9` → 5 lẻ, 4 chẵn → hiệu `5 - 4 = 1`.
- Mỗi chữ số sau: `0..9` → 5 lẻ, 5 chẵn → hiệu `0`.

`T = 9 · 10^(d-1)`, `D = 1 · 0^(d-1)` = `1` nếu `d = 1`, `0` nếu `d >= 2`.

→ Với `d >= 2`, số tốt là **đúng `T / 2`** — giải thích quan sát ở mục 3! (Chữ số tự do có 5 chẵn, 5 lẻ, "tung đồng xu công bằng".) Với `d = 1`: `(9 + 1) / 2 = 5` (`1, 3, 5, 7, 9`).

## 6.3. Đếm `F(x)` bằng tổ hợp

- Các số có **ít chữ số hơn** `x`: cộng theo công thức 6.2.
- Các số **cùng độ dài** `<= x`: duyệt từng vị trí `i` của `x`; tại `i`, thử mọi chữ số `d` **nhỏ hơn** `x[i]` (chữ số đầu thì từ 1). Phần còn lại `rem` chữ số tự do → dùng `(T + D) / 2` với tính chẵn lẻ đã có. Rồi "đi theo" chữ số `x[i]`. Cuối cùng xét chính `x`.

In [7]:
def F_combinatorial(x: int) -> int:
    if x <= 0:
        return 0
    s = str(x)
    L = len(s)
    total = 0
    for d in range(1, L):                                   # các số ngắn hơn
        T = 9 * 10 ** (d - 1)
        D = 1 if d == 1 else 0
        total += (T + D) // 2
    par = 0                                                 # tính chẵn lẻ của tiền tố đi theo x
    for i, ch in enumerate(s):
        c = int(ch)
        rem = L - i - 1
        for d in range(1 if i == 0 else 0, c):              # chữ số nhỏ hơn x[i]
            p = par ^ (1 if d % 2 == 0 else 0)
            T = 10 ** rem
            D = (1 if rem == 0 else 0) * (1 if p == 0 else -1)   # tổng có dấu, nhân thêm dấu của tiền tố
            total += (T + D) // 2
        par ^= 1 if c % 2 == 0 else 0
    return total + (1 if par == 0 else 0)                   # chính x


print(F_combinatorial(22), F_combinatorial(17), F_combinatorial(10**15))

12 9 500000000000000


(Ở đây `D = ±1` chỉ khi `rem = 0` vì mọi chữ số tự do có hiệu `5 - 5 = 0`; dấu `-1` khi tiền tố đã có số chữ số chẵn lẻ, vì khi đó "tổng chẵn" ứng với "phần còn lại lẻ".)

# 7. Phiên bản đúng format LeetCode

Ta nộp digit DP (cách 1) — tổng quát, dễ sửa cho biến thể.

In [8]:
from functools import lru_cache


class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
        def F(x: int) -> int:
            if x <= 0:
                return 0
            digits = list(map(int, str(x)))
            n = len(digits)

            @lru_cache(maxsize=None)
            def go(i, par, tight, started):
                if i == n:
                    return 1 if started and par == 0 else 0
                limit = digits[i] if tight else 9
                total = 0
                for d in range(limit + 1):
                    ns = started or d > 0
                    total += go(i + 1, par ^ (1 if ns and d % 2 == 0 else 0), tight and d == limit, ns)
                return total

            return go(0, 0, True, False)

        return F(r) - F(l - 1)

In [9]:
sol = Solution()
tests = [
    (18, 22, 3, "ví dụ 1"),
    (98, 101, 2, "ví dụ 2"),
    (1, 10, 5, "ví dụ 3"),
    (1, 1, 1, "một số lẻ"),
    (2, 2, 0, "một chữ số chẵn"),
    (10, 99, 45, "đúng một nửa số có 2 chữ số"),
    (1, 10**15, 500000000000000, "cực lớn"),
]
for l, r, expected, name in tests:
    result = sol.countEvenlyGoodIntegers(l, r)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:28} -> {result}")

PASS | ví dụ 1                      -> 3
PASS | ví dụ 2                      -> 2
PASS | ví dụ 3                      -> 5
PASS | một số lẻ                    -> 1
PASS | một chữ số chẵn              -> 0
PASS | đúng một nửa số có 2 chữ số  -> 45
PASS | cực lớn                      -> 500000000000000


Test cuối: `[1, 10^15]` gồm các số 1..15 chữ số (đúng một nửa từ 2 chữ số trở lên, 5 số một chữ số) cộng thêm `10^15` (16 chữ số: một số 1 và **15** số 0 → 15 chữ số chẵn, lẻ → không tốt). Tổng: `5 + Σ_{d=2}^{15} 4.5·10^(d-1) = 500000000000000`.

# 8. Vì sao đúng?

1. **`F(r) - F(l-1)`**: tập `[l, r] = [1, r] \ [1, l-1]`.
2. **Digit DP**: mỗi số `y` trong `[0, x]` ứng với đúng một dãy chữ số (độ dài bằng `x`, có số 0 ở đầu) không vượt `x` theo thứ tự từ điển. Cờ `tight` đảm bảo không vượt; cờ `started` loại số 0 ở đầu khỏi việc đếm chữ số chẵn và loại chính số 0.
3. **Tổ hợp**: khai triển `Π (a_i + b_i·t)` tại `t = 1` và `t = -1` cho tổng số cách và tổng có dấu; trung bình cộng là số cách có số chữ số chẵn là chẵn.

# 9. Độ phức tạp

Gọi `D` = số chữ số của `r` (≤ 16).

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | duyệt mọi số | `O((r - l) · D)` | `O(1)` |
| 2. Digit DP (đệ quy / lặp) | trạng thái `(i, par, tight, started)` | `O(D · 2·2·2 · 10)` | `O(D)` |
| 3. Tổ hợp | `(T + D) / 2` theo vị trí | `O(D · 10)` | `O(1)` |

# 10. Đối chiếu bốn cách

In [10]:
import random

for x in range(0, 3000):
    b = count_bruteforce(1, x) if x > 0 else 0
    assert F_digit_dp(x) == b and F_iterative(x) == b and F_combinatorial(x) == b, x
random.seed(4060)
for _ in range(500):
    l = random.randint(1, 10**15)
    r = random.randint(l, 10**15)
    a = count_digit_dp(l, r)
    assert a == F_iterative(r) - F_iterative(l - 1) == F_combinatorial(r) - F_combinatorial(l - 1)
    assert a == sol.countEvenlyGoodIntegers(l, r)
print("Khớp brute force với mọi x < 3000; các cách nhanh khớp nhau trên 500 khoảng ngẫu nhiên tới 10^15")

Khớp brute force với mọi x < 3000; các cách nhanh khớp nhau trên 500 khoảng ngẫu nhiên tới 10^15


# 11. Những lỗi dễ mắc

## Lỗi 1 — Đếm số 0 ở đầu là chữ số chẵn

Nếu bỏ cờ `started`, số `7` (viết là `07` khi cận trên có 2 chữ số) bị coi là có một chữ số chẵn.

In [11]:
def wrong_no_started(x):
    digits = list(map(int, str(x)))
    n = len(digits)

    @lru_cache(maxsize=None)
    def go(i, par, tight):
        if i == n:
            return 1 if par == 0 else 0
        limit = digits[i] if tight else 9
        return sum(go(i + 1, par ^ (d % 2 == 0), tight and d == limit) for d in range(limit + 1))

    return go(0, 0, True) - 1        # trừ số 0 (cũng đã sai)


print("Sai :", wrong_no_started(10), "(đếm trên [1, 10])")
print("Đúng:", sol.countEvenlyGoodIntegers(1, 10))

Sai : 4 (đếm trên [1, 10])
Đúng: 5


## Lỗi 2 — Dùng `F(r) - F(l)` thay vì `F(r) - F(l - 1)`

Làm mất chính số `l` nếu nó tốt.

In [12]:
print("Sai :", F_digit_dp(22) - F_digit_dp(19), "| Đúng:", sol.countEvenlyGoodIntegers(19, 22))

Sai : 2 | Đúng: 3


## Lỗi 3 — Quên cập nhật `tight` đúng cách

`tight` mới = `tight and d == limit`. Viết `d == limit` (bỏ `tight and`) sẽ chặn sai khi tiền tố đã nhỏ hơn `x` (khi đó `limit = 9` và chọn `9` không có nghĩa là còn bám sát `x`).

## Lỗi 4 — Ghi nhớ (memo) theo trạng thái mà quên `tight`/`started`

Các trạng thái khác `tight` hoặc `started` cho kết quả khác nhau — đưa đủ vào tham số của hàm được `lru_cache`.

## Lỗi 5 — Brute Force với `r = 10^15`

Không bao giờ xong. Thấy "đếm số trong `[l, r]` với `r` tới `10^15`/`10^18`, tính chất về chữ số" → digit DP.

# 12. Mẹo nhận diện

1. Đếm số nguyên trong `[l, r]` (rất lớn) thỏa tính chất **về chữ số** → **digit DP** + `F(r) - F(l - 1)`.
2. Trạng thái tối thiểu: vị trí, cờ `tight`, (cờ `started` nếu số 0 ở đầu ảnh hưởng), và **thông tin cần nhớ** về các chữ số đã điền (ở đây chỉ 1 bit chẵn/lẻ).
3. Tính chất "số lượng X là chẵn/lẻ" → mẹo `(T ± D) / 2` với tổng có dấu.

Các bài liên quan: *Number of Digit One* (LeetCode 233), *Numbers With Repeated Digits* (1012), *Count Special Integers* (2376), *Count Stepping Numbers in Range* (2801).

# 13. Biến thể

## Biến thể 1 — Số chữ số chẵn chia hết cho `m`

Thay bit chẵn/lẻ bằng `cnt mod m`.

In [13]:
def count_even_digits_mod(l: int, r: int, m: int) -> int:
    def F(x):
        if x <= 0:
            return 0
        digits = list(map(int, str(x)))
        n = len(digits)

        @lru_cache(maxsize=None)
        def go(i, c, tight, started):
            if i == n:
                return 1 if started and c == 0 else 0
            limit = digits[i] if tight else 9
            total = 0
            for d in range(limit + 1):
                ns = started or d > 0
                total += go(i + 1, (c + (1 if ns and d % 2 == 0 else 0)) % m, tight and d == limit, ns)
            return total

        return go(0, 0, True, False)

    return F(r) - F(l - 1)


print(count_even_digits_mod(1, 1000, 3),
      sum(1 for x in range(1, 1001) if count_even_digits(x) % 3 == 0))

256 256


## Biến thể 2 — Số chữ số chẵn **bằng** số chữ số lẻ

Nhớ hiệu `(số chẵn - số lẻ)` (lệch tối đa ±16) thay cho 1 bit. (Bài tập 6.)

# 14. Template tư duy

```python
def F(x):                       # đếm trong [1, x]
    digits = list(map(int, str(x)))
    @lru_cache(None)
    def go(i, state, tight, started):
        if i == len(digits): return int(started and ok(state))
        limit = digits[i] if tight else 9
        return sum(go(i + 1, update(state, d, started or d > 0), tight and d == limit, started or d > 0)
                   for d in range(limit + 1))
    return go(0, init, True, False)
answer = F(r) - F(l - 1)
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `F(105)` (số tốt trong `[1, 105]`) bằng cách chia nhóm: số 1 chữ số, số 2 chữ số, số 3 chữ số `100..105`. Kiểm tra bằng cell dưới.

In [14]:
# Kiểm tra Bài 1
print(F_digit_dp(105), [y for y in range(100, 106) if is_evenly_good(y)])

53 [100, 102, 104]


## Bài 2 — Dự đoán output

```python
print(count_even_digits(2024), is_evenly_good(2024), is_evenly_good(13579))
print(F_digit_dp(9), F_digit_dp(99))
```

In [15]:
print(count_even_digits(2024), is_evenly_good(2024), is_evenly_good(13579))
print(F_digit_dp(9), F_digit_dp(99))

4 True True
5 50


## Bài 3 — Hoàn thiện digit DP

In [16]:
def exercise_3_F(x: int) -> int:
    # TODO: go(i, par, tight, started) như mục 4
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_F(22) == 12

## Bài 4 — Đếm số không chứa chữ số 4 trong `[1, x]`

```python
no_four(50)   -> 36     # bỏ 4, 14, 24, 34 và 40..49 (14 số)
```

In [17]:
def exercise_4_no_four(x: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_no_four(50) == 36

## Bài 5 — Count Special Integers (LeetCode 2376)

Đếm số trong `[1, n]` có **mọi chữ số khác nhau**.

```python
special(20)   -> 19     # chỉ 11 bị loại
special(135)  -> 110
```

In [18]:
def exercise_5_special(n: int) -> int:
    # TODO: trạng thái thêm bitmask các chữ số đã dùng
    raise NotImplementedError


# assert exercise_5_special(20) == 19
# assert exercise_5_special(135) == 110

## Bài 6 — Mini challenge: số chữ số chẵn bằng số chữ số lẻ

Đếm số trong `[l, r]` (tới `10^15`) có số chữ số chẵn **bằng** số chữ số lẻ.

```python
balanced(1, 100)   -> 45    # các số 2 chữ số có một chẵn một lẻ
```

In [19]:
def exercise_6_balanced(l: int, r: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_balanced(1, 100) == 45

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `limit = digits[i] if tight else 9`.
- Mức 2: `ns = started or d > 0`; chữ số chẵn chỉ tính khi `ns`.
- Mức 3: kết thúc trả `1` nếu `started and par == 0`.

**Bài 4**

- Mức 1: bỏ qua `d == 4` trong vòng lặp chữ số.
- Mức 2: không cần `par`, chỉ cần `tight`, `started`.
- Mức 3: kết thúc trả `1` nếu `started`.

**Bài 5**

- Mức 1: thêm `mask` các chữ số đã dùng (chỉ tính sau khi `started`).
- Mức 2: bỏ qua `d` nếu `started` và bit `d` đã có trong `mask`.
- Mức 3: số 0 ở đầu không đưa vào `mask`.

**Bài 6**

- Mức 1: nhớ `diff = (#chẵn) - (#lẻ)`, cộng thêm offset 20 cho không âm.
- Mức 2: chữ số chỉ tính khi `started`.
- Mức 3: kết thúc trả `1` nếu `started and diff == 0`.

# 16. Đáp án tham khảo

## Đáp án Bài 1

- 1 chữ số: `1, 3, 5, 7, 9` → 5.
- 2 chữ số: đúng một nửa của 90 → 45.
- `100..105`: `100` (hai số 0) ✔, `101` (một số 0) ✗, `102` (0, 2) ✔, `103` ✗, `104` (0, 4) ✔, `105` ✗ → 3.

`F(105) = 5 + 45 + 3 = 53`.

## Đáp án Bài 2

```text
4 True True
5 50
```

In [20]:
# Đáp án Bài 3
def answer_3_F(x: int) -> int:
    return F_digit_dp(x)


assert answer_3_F(22) == 12
assert answer_3_F(105) == 53
print("Bài 3: PASS")

Bài 3: PASS


In [21]:
# Đáp án Bài 4
def answer_4_no_four(x: int) -> int:
    digits = list(map(int, str(x)))
    n = len(digits)

    @lru_cache(maxsize=None)
    def go(i, tight, started):
        if i == n:
            return 1 if started else 0
        limit = digits[i] if tight else 9
        return sum(go(i + 1, tight and d == limit, started or d > 0) for d in range(limit + 1) if d != 4)

    return go(0, True, False)


assert answer_4_no_four(50) == 36
assert all(answer_4_no_four(x) == sum("4" not in str(y) for y in range(1, x + 1)) for x in range(1, 500))
print("Bài 4: PASS")

Bài 4: PASS


In [22]:
# Đáp án Bài 5
def answer_5_special(n: int) -> int:
    digits = list(map(int, str(n)))
    L = len(digits)

    @lru_cache(maxsize=None)
    def go(i, mask, tight, started):
        if i == L:
            return 1 if started else 0
        limit = digits[i] if tight else 9
        total = 0
        for d in range(limit + 1):
            if started and mask >> d & 1:
                continue
            ns = started or d > 0
            total += go(i + 1, mask | (1 << d) if ns else mask, tight and d == limit, ns)
        return total

    return go(0, 0, True, False)


assert answer_5_special(20) == 19
assert answer_5_special(135) == 110
print("Bài 5: PASS")

Bài 5: PASS


In [23]:
# Đáp án Bài 6
def answer_6_balanced(l: int, r: int) -> int:
    def F(x):
        if x <= 0:
            return 0
        digits = list(map(int, str(x)))
        n = len(digits)

        @lru_cache(maxsize=None)
        def go(i, diff, tight, started):
            if i == n:
                return 1 if started and diff == 0 else 0
            limit = digits[i] if tight else 9
            total = 0
            for d in range(limit + 1):
                ns = started or d > 0
                nd = diff + ((1 if d % 2 == 0 else -1) if ns else 0)
                total += go(i + 1, nd, tight and d == limit, ns)
            return total

        return go(0, 0, True, False)

    return F(r) - F(l - 1)


def brute_balanced(l, r):
    return sum(1 for x in range(l, r + 1) if 2 * count_even_digits(x) == len(str(x)))


assert answer_6_balanced(1, 100) == 45
assert answer_6_balanced(1, 5000) == brute_balanced(1, 5000)
print("Bài 6: PASS | [1, 10^15]:", answer_6_balanced(1, 10**15))

Bài 6: PASS | [1, 10^15]: 19057806144045


# 17. Tổng kết

**Count Evenly Good Integers**:

1. `F(r) - F(l - 1)` → chỉ cần đếm trên `[1, x]`.
2. Digit DP: điền chữ số từ trái sang phải, trạng thái `(i, par, tight, started)`.
3. Mẹo `(T + D) / 2`: với số từ 2 chữ số trở lên, đúng **một nửa** là tốt (chữ số tự do có 5 chẵn, 5 lẻ).

## Độ phức tạp

```text
Time:  O(D · 10 · 8)  với D ≤ 16
Space: O(D)
```

# 18. Checklist trước khi submit

- [ ] Dùng `F(r) - F(l - 1)`, với `F(0) = 0`?
- [ ] Có cờ `started` để số 0 ở đầu không bị tính là chữ số chẵn, và không đếm số 0?
- [ ] `tight` mới = `tight and d == limit`?
- [ ] `lru_cache` theo đủ các tham số trạng thái (memo tạo mới cho mỗi `x`)?
- [ ] Đã test `l = r`, số một chữ số chẵn/lẻ, khoảng cực lớn?